In [1]:
# =========================================================
# 1. Imports
# =========================================================
import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

# =========================================================
# 2. Caminhos
# =========================================================

BASE = Path("/Users/fernandogomes/MacLab/mhac")

favela_fp = BASE / "data/geosampa/SIRGAS_GPKG_favela.gpkg"

# mosaico LiDAR (ajuste conforme seu nome real)
raster_fp = BASE / "data/city_mosaics/hag_2020_mosaic_nodata.tif"
# pode usar também:
# mds_2020_mosaic.tif

# =========================================================
# 3. Carregar dados
# =========================================================

favelas = gpd.read_file(favela_fp)

with rasterio.open(raster_fp) as src:
    raster = src.read(1)
    affine = src.transform
    nodata = src.nodata

print(favelas.crs)
print(src.crs)

/Users/fernandogomes/miniconda3/envs/pdal/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


EPSG:31983
EPSG:31983


In [2]:
# =========================================================
# 4. Criar máscara de área construída
# =========================================================

# regra simples (ajuste se quiser):
# HAG > 2m ≈ edificação

built_mask = np.where((raster > 2) & (raster != nodata), 1, 0)

# salvar temporariamente (opcional, mas ajuda no zonal_stats)
tmp_raster = BASE / "tmp_built_mask.tif"

with rasterio.open(
    tmp_raster,
    "w",
    driver="GTiff",
    height=built_mask.shape[0],
    width=built_mask.shape[1],
    count=1,
    dtype="uint8",
    crs=src.crs,
    transform=affine,
    nodata=0,
) as dst:
    dst.write(built_mask, 1)

print("Raster binário criado.")

Raster binário criado.


In [6]:
# =========================================================
# 5. Área construída dentro das favelas
# =========================================================

stats = zonal_stats(
    favelas,
    tmp_raster,
    stats=["sum"],
    nodata=0
)

# cada pixel = 1 m² (se seu raster for 1m)
favelas["built_area_m2"] = [s["sum"] for s in stats]

area_favelas = favelas["built_area_m2"].sum()

print(f"Área construída nas favelas: {area_favelas/1e6:.2f} km²")

Área construída nas favelas: 20.52 km²


In [9]:
# =========================================================
# 6. Área construída total da cidade
# =========================================================

total_built_pixels = np.sum(built_mask)

area_total = total_built_pixels  # em m² (se pixel = 1m)

print(f"Área construída total: {area_total/1e6:.2f} km²")

Área construída total: 669.39 km²


In [10]:
# =========================================================
# 7. Percentual
# =========================================================

percent = (area_favelas / area_total) * 100

print(f"Percentual: {percent:.2f}%")


Percentual: 3.07%


In [13]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

BASE = Path("/Users/fernandogomes/MacLab/mhac")

quadras_fp = BASE / "data/analises/quadras_cadastro_vs_lidar_2020.gpkg"
favela_fp  = BASE / "data/geosampa/SIRGAS_GPKG_favela.gpkg"

# --------------------------------------------------
# 1. Ler dados
# --------------------------------------------------
quadras = gpd.read_file(quadras_fp)
favelas = gpd.read_file(favela_fp)

print("CRS quadras:", quadras.crs)
print("CRS favelas:", favelas.crs)

if quadras.crs != favelas.crs:
    favelas = favelas.to_crs(quadras.crs)

# --------------------------------------------------
# 2. Limpeza básica das geometrias
# --------------------------------------------------
quadras = quadras[quadras.geometry.notnull()].copy()
favelas = favelas[favelas.geometry.notnull()].copy()

quadras = quadras[~quadras.geometry.is_empty].copy()
favelas = favelas[~favelas.geometry.is_empty].copy()

# corrigir geometrias inválidas
quadras["geometry"] = quadras.geometry.buffer(0)
favelas["geometry"] = favelas.geometry.buffer(0)

quadras = quadras[quadras.is_valid].copy()
favelas = favelas[favelas.is_valid].copy()

# manter apenas polígonos
quadras = quadras[quadras.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
favelas = favelas[favelas.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

print("Quadras:", len(quadras))
print("Favelas:", len(favelas))

# --------------------------------------------------
# 3. Garantir área da quadra e denominador
# --------------------------------------------------
quadras["area_quadra_m2"] = quadras.geometry.area

# confere nome da coluna
print(quadras.columns)

total_area_ocupada = quadras["area_ocupada_m2"].sum()
print(f"Área ocupada total nas quadras (2020): {total_area_ocupada/1e6:.2f} km²")

# --------------------------------------------------
# 4. Overlay
# --------------------------------------------------
inter = gpd.overlay(
    quadras[["SQ", "area_ocupada_m2", "area_quadra_m2", "geometry"]],
    favelas[["geometry"]],
    how="intersection",
    keep_geom_type=False
)

# depois do overlay, manter só polígonos
inter = inter[inter.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

print("Interseções:", len(inter))

# --------------------------------------------------
# 5. Ponderação
# --------------------------------------------------
inter["area_intersection_m2"] = inter.geometry.area
inter["ratio"] = inter["area_intersection_m2"] / inter["area_quadra_m2"]

# evitar razões > 1 por ruído numérico
inter["ratio"] = inter["ratio"].clip(lower=0, upper=1)

inter["area_ocupada_favela_m2"] = inter["area_ocupada_m2"] * inter["ratio"]

favela_area_ocupada = inter["area_ocupada_favela_m2"].sum()
percentual = 100 * favela_area_ocupada / total_area_ocupada

print(f"Área ocupada estimada dentro de favelas: {favela_area_ocupada/1e6:.2f} km²")
print(f"Percentual: {percentual:.2f}%")

/Users/fernandogomes/miniconda3/envs/pdal/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


CRS quadras: EPSG:31983
CRS favelas: EPSG:31983
Quadras: 45371
Favelas: 2163
Index(['qd_fiscal_cad', 'qd_id_cad', 'qd_setor_cad', 'SQ', 'setor', 'quadra',
       'area_terreno_m2', 'area_construida_m2', 'area_ocupada_m2',
       'pavimentos_min', 'pavimentos_max', 'ano', 'year', 'qd_id_lidar',
       'qd_setor_lidar', 'qd_fiscal_lidar', 'area_m2', 'count_total', 'min',
       'max', 'mean', 'count_valid', 'sum', 'std', 'median', 'valid_frac',
       'nodata_frac', 'valid_area_m2', 'raster', 'height_threshold_m',
       'geometry', 'area_quadra_m2'],
      dtype='object')
Área ocupada total nas quadras (2020): 228.72 km²
Interseções: 3734
Área ocupada estimada dentro de favelas: 2.68 km²
Percentual: 1.17%
